# Predicting Stock Prices With RNNs

Approach:
- Using stock prices from the Energy industry (Tickers: BP, COP, CVX, DVN, SHEL, SLB, XOM)
- Predicting log returns rather than price for the statistical benefits of a stationary time series and homoskedasticity
- Also using FRED macroeconomic data 

In [95]:
import pandas as pd

bp = pd.read_csv("../datasets/energy_stock_data/bp.us.txt")
cop = pd.read_csv("../datasets/energy_stock_data/cop.us.txt")
cvx = pd.read_csv("../datasets/energy_stock_data/cvx.us.txt")
dvn = pd.read_csv("../datasets/energy_stock_data/dvn.us.txt")
slb = pd.read_csv("../datasets/energy_stock_data/slb.us.txt")
xom = pd.read_csv("../datasets/energy_stock_data/xom.us.txt")


In [96]:
import os
from dotenv import load_dotenv
from fredapi import Fred

load_dotenv()
fred_key = os.getenv("FRED_KEY")
f = Fred(fred_key)

series_ids = {
    'unemployment': 'UNRATE',
    'cpi': 'CPIAUCSL',
    'gasoline': 'GASREGCOVW',
    'fed_funds': 'FEDFUNDS',
}


dfs = []
for name, sid in series_ids.items():
    releases = f.get_series_all_releases(sid)
    first_release = releases.groupby('date').first().reset_index()
    first_release = first_release.rename(columns={'value': name, 'realtime_start': f'{name}_release_date'})
    first_release = first_release.set_index('date')
    dfs.append(first_release)

macro = pd.concat(dfs, axis=1)
macro = macro.resample('MS').last()
macro.index = pd.to_datetime(macro.index)
macro

/tmp/ipykernel_39133/1268428875.py:25: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  macro = pd.concat(dfs, axis=1)


,unemployment_release_date,unemployment,cpi_release_date,cpi,gasoline_release_date,gasoline,fed_funds_release_date,fed_funds
date,,,,,,,,
1947-01-01,None,None,1994-02-17 00:00:00,21.48,None,None,None,None
1947-02-01,None,None,1994-02-17 00:00:00,21.62,None,None,None,None
1947-03-01,None,None,1994-02-17 00:00:00,22.0,None,None,None,None
1947-04-01,None,None,1994-02-17 00:00:00,22.0,None,None,None,None
1947-05-01,None,None,1994-02-17 00:00:00,21.95,None,None,None,None
...,...,...,...,...,...,...,...,...
2025-11-01,2025-12-16 00:00:00,4.6,2025-12-18 00:00:00,325.031,2025-11-25 00:00:00,2.936,2025-12-01 00:00:00,3.88
2025-12-01,2026-01-09 00:00:00,4.4,2026-01-13 00:00:00,326.03,2025-12-30 00:00:00,2.69,2026-01-02 00:00:00,3.72
2026-01-01,2026-02-11 00:00:00,4.3,2026-02-13 00:00:00,326.588,2026-01-27 00:00:00,2.747,2026-02-02 00:00:00,3.64


In [97]:
tickers = {'bp': bp, 'cop': cop, 'cvx': cvx, 'dvn': dvn, 'slb': slb, 'xom': xom}

dfs = []
for name, df in tickers.items():
    tmp = df[['<DATE>', '<CLOSE>', '<VOL>']].copy()
    tmp.columns = ['date', f'{name}_close', f'{name}_vol']
    tmp['date'] = pd.to_datetime(tmp['date'], format='%Y%m%d')
    tmp = tmp.set_index('date')
    dfs.append(tmp)

combined = pd.concat(dfs, axis=1)
combined.reset_index(inplace=True)
combined['date'] = pd.to_datetime(combined['date'])
combined = combined.sort_values('date')
macro_reset = macro[['unemployment', 'cpi', 'gasoline', 'fed_funds']].reset_index().sort_values('date')

combined = pd.merge_asof(combined, macro_reset, on='date', direction='backward', allow_exact_matches=False)

combined 

/tmp/ipykernel_39133/3866546309.py:11: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  combined = pd.concat(dfs, axis=1)


,date,bp_close,bp_vol,cop_close,cop_vol,cvx_close,cvx_vol,dvn_close,dvn_vol,slb_close,slb_vol,xom_close,xom_vol,unemployment,cpi,gasoline,fed_funds
0,1970-01-02,NaN,NaN,NaN,NaN,0.599556,1.053063e+05,NaN,NaN,NaN,NaN,1.15116,1.898333e+06,3.9,37.9,None,8.98
1,1970-01-05,NaN,NaN,NaN,NaN,0.605993,2.243766e+05,NaN,NaN,NaN,NaN,1.17625,3.041474e+06,3.9,37.9,None,8.98
2,1970-01-06,NaN,NaN,NaN,NaN,0.593510,1.107535e+05,NaN,NaN,NaN,NaN,1.17012,1.991447e+06,3.9,37.9,None,8.98
3,1970-01-07,NaN,NaN,NaN,NaN,0.593510,9.026368e+04,NaN,NaN,NaN,NaN,1.16390,1.484532e+06,3.9,37.9,None,8.98
4,1970-01-08,NaN,NaN,NaN,NaN,0.605993,1.270725e+05,NaN,NaN,NaN,NaN,1.16390,1.737983e+06,3.9,37.9,None,8.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14168,2026-03-16,42.90,15163107.0,121.32,7556780.0,196.840000,1.268392e+07,46.65,19839741.0,44.96,12648472.0,157.23000,2.283563e+07,None,None,3.566,None
14169,2026-03-17,43.85,18098354.0,122.87,7953143.0,197.970000,1.401858e+07,47.42,18441738.0,46.13,14766158.0,158.81000,2.159217e+07,None,None,3.566,None
14170,2026-03-18,44.61,16789125.0,123.65,10273029.0,198.610000,1.330500e+07,48.16,21007901.0,45.32,13276982.0,157.59000,1.894714e+07,None,None,3.566,None
14171,2026-03-19,45.86,37659746.0,126.02,11205373.0,201.440000,1.691843e+07,48.79,28297069.0,47.82,28157955.0,158.16000,2.708273e+07,None,None,3.566,None


In [103]:
combined[["unemployment", "cpi", "gasoline", "fed_funds"]] = combined[["unemployment", "cpi", "gasoline", "fed_funds"]].ffill(inplace=False)
combined.loc[combined.date > "2000-01-01"].isna().sum()

date               0
bp_close        1294
bp_vol          1294
cop_close          3
cop_vol            3
cvx_close          2
cvx_vol            2
dvn_close          0
dvn_vol            0
slb_close          0
slb_vol            0
xom_close          1
xom_vol            1
unemployment       0
cpi                0
gasoline           0
fed_funds          0
dtype: int64

In [105]:
combined.loc[(combined.cop_close.isna()) & (combined.date > "2000-01-01")]

,date,bp_close,bp_vol,cop_close,cop_vol,cvx_close,cvx_vol,dvn_close,dvn_vol,slb_close,slb_vol,xom_close,xom_vol,unemployment,cpi,gasoline,fed_funds
8236,2002-08-15,NaN,NaN,NaN,NaN,23.8019,9.678454e+06,17.6403,2.577142e+06,16.1566,1.166093e+07,22.9773,1.798408e+07,5.7,180.5,1.365,1.74
8257,2002-09-16,NaN,NaN,NaN,NaN,22.6304,6.723734e+06,18.1768,1.002470e+06,15.7810,5.402176e+06,21.2843,1.229231e+07,5.6,180.8,1.385,1.75
10318,2010-11-19,21.722,1.004017e+07,NaN,NaN,NaN,NaN,54.5726,7.558638e+06,57.1420,1.025477e+07,43.9009,3.874595e+07,9.8,219.146,2.805,0.19


# next steps:

manually fill in the few missing vals

start building models